# Mbrain training

This public notebook was migrated from the audited read-only research source. 
Configure paths in `config.yaml` before execution. Time is metadata and is never a model input.


In [ ]:
from pathlib import Path
import yaml

DATASET = 'Mbrain'
CWD = Path.cwd().resolve()
EXPERIMENT_DIR = CWD if (CWD / "config.yaml").is_file() else CWD / "experiments" / DATASET
REPO_ROOT = EXPERIMENT_DIR.parents[1]
import sys
sys.path.append(str(REPO_ROOT / "src"))

CONFIG = yaml.safe_load(
    (EXPERIMENT_DIR / "config.yaml").read_text(encoding="utf-8")
)

def experiment_path(value):
    path = Path(value)
    return path if path.is_absolute() else (EXPERIMENT_DIR / path).resolve()

DATA_ROOT = experiment_path(CONFIG["data_root"])
RUN_ROOT = experiment_path(CONFIG["run_root"])
CHECKPOINT_ROOT = experiment_path(CONFIG["checkpoint_root"])
SCANVI_DIR = experiment_path(CONFIG["scanvi_dir"])
SCANVI_ADATA = SCANVI_DIR / "adata.h5ad"
STAGE1_CHECKPOINT_ROOT = experiment_path(CONFIG["stage1_checkpoint"])
STAGE2_CHECKPOINT_ROOT = experiment_path(CONFIG["stage2_checkpoint"])
LR_PAIRS = experiment_path(CONFIG["lr_pairs_path"])
DIFF_MAP = experiment_path(CONFIG["diff_map_path"]) if "diff_map_path" in CONFIG else None
DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)


In [ ]:
import re, os
import scanpy as sc
import torch
import numpy as np
import pandas as pd
import sys

import scvi
from pathlib import Path
import importlib
from stvirtual.models import stage1_2d as s1
from stvirtual.models import stage2_2d as s2


In [ ]:
data_path = str(DATA_ROOT)
ckpt_path = str(CHECKPOINT_ROOT)
resl_path = str(RUN_ROOT)
lrpr_path = str(LR_PAIRS)


In [ ]:
route_ids=['T170', 'T171']
fracs = ["T170_to_T171"]
seg_key = "T170_to_T171" 
steps = 10


In [ ]:
adata = sc.read_h5ad(SCANVI_ADATA)
adata


AnnData object with n_obs × n_vars = 138384 × 20161
    obs: 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'region', 'coor_x', 'coor_y', 'raw_x', 'raw_y', 'distance', 'area', 'Astrocyte', 'Bergmann', 'Choroid', 'Endothelial_mural', 'Endothelial_stalk', 'Ependymal', 'Fibroblast', 'Golgi', 'Granule', 'MLI1', 'MLI2', 'Macrophage', 'Microglia', 'ODC', 'OPC', 'PLI', 'Purkinje', 'UBC', 'cluster', 'annotation', 'sample', 'n_genes', 'cx_aligned', 'cy_aligned', 'cx_aligned_norm', 'cy_aligned_norm', 'LR_potential', 'LR_potential_z', '_scvi_batch', '_scvi_labels', 'leiden_scVI'
    var: 'n_counts', 'highly_variable', 'highly_variable_rank', 'means', 'variances', 'variances_norm', 'highly_variable_nbatches'
    uns: '_scvi_manager_uuid', '_scvi_uuid', 'alignment_reference', 'annotation_colors', 'hvg', 'leiden_scVI', 'leiden_scVI_colors', 'log1p', 'neighbors', 'sample_colors', 'umap'
    obsm: 'X_scanVI', 'X_umap', 'spatial', 'spatial_aligned'
    layers: 'counts'
    obsp: 'connectivities', 'distanc

In [ ]:
adata.obs['sample'].unique()


['T168', 'T169', 'T170', 'T171']
Categories (4, object): ['T168', 'T169', 'T170', 'T171']

In [ ]:
model = None


In [ ]:
importlib.reload(s1)

res1 = s1.train_model_multislice(
    model=model,                  
    adata_all=adata,
    slice_key="sample",
    route_ids=route_ids,
    save_root=str(STAGE1_CHECKPOINT_ROOT),
    x_key="cx_aligned",
    y_key="cy_aligned",
    latent_key="X_scanVI",    
    steps=steps,  
    guide_eps=0.005,
    uot_eps=0.05, uot_tau=1.0 , uot_lam_x=2.0, uot_lam_f=0.1,
    guide_topk=512, guide_temp=0.5, guide_schedule="linear",
    cell_type_key='annotation',
    lam_context=5.0,
    lam_residual=0.1,
    lam_vsmooth=0.1,  
    lam_uot=1.0,
    lib_layer='counts',              
    latent_dim=10,             
    epochs=100,
    device="cuda:0",
)



[MultiSlice] Train segment: T170_to_T171 (n_src=34874, n_tgt=46100)
  Global norm route_ids=['T170', 'T171']  latent_dim=10

Pre-computing neighbor context features...
[ctx] scale 1/1: sparse matmul ...  (E=557984)
[ctx] scale 1/1: sparse matmul ...  (E=737600)


Train (NeuralODE dopri5): 100%|██████████| 100/100 [02:00<00:00,  1.21s/it, loss=0.0121, uot=0.0082, vs=0.0379]


In [ ]:
# saved = {'T170_to_T171': str(CHECKPOINT_ROOT / 'stage1_res' / 'rollout_stage1_T170_to_T171.npz')}


## Save Stage-1 traces and generate 2D boundaries

In [ ]:
from stvirtual.utils import boundary_2d as boundary
from stvirtual.utils.stage1_results import save_res as save_stage1_results
from stvirtual.utils.trajectory import rollout_trace_from_out

boundary_cfg = CONFIG["boundary"]
saved = save_stage1_results(
    res1=res1, adata_all=adata,
    out_dir=str(CHECKPOINT_ROOT / "stage1_res"),
    slice_key="sample", ann_key="annotation",
    save_prefix="rollout_stage1", steps=steps,
    n_cache=boundary_cfg["n_cache"], unnormalize=False,
)

for stage_key in fracs:
    source_name = stage_key.split("_to_", 1)[0]
    match = re.search(r"\d+", source_name)
    source_number = int(match.group(0)) if match else 10**9
    trace = rollout_trace_from_out(res1[stage_key], steps=steps, n_cache=boundary_cfg["n_cache"], unnormalize=False)
    frames = [value.detach().cpu().numpy().astype(np.float32) for value in trace["x"]]
    stage_dir = RUN_ROOT / "bound" / stage_key
    stage_dir.mkdir(parents=True, exist_ok=True)
    statistics = []
    for frame_index, coordinates in enumerate(frames):
        shell, fraction, outside, expansion = boundary.make_shell_adaptive(
            coordinates, alpha_factor=boundary_cfg["alpha_factor"],
            seed=2026 + source_number + frame_index,
            n_resample=boundary_cfg["n_resample"], target_frac=boundary_cfg["target_frac"],
            expand0=boundary_cfg["expand0"], expand_step=boundary_cfg["expand_step"],
            expand_max=boundary_cfg["expand_max"], fallback_expand=boundary_cfg["fallback_expand"],
        )
        boundary.save_shell_csv(shell, stage_dir / f"bound_z{frame_index:03d}.csv")
        boundary.plot_shell_coverage(coordinates, shell, outside, stage_dir / f"bound_z{frame_index:03d}.png")
        statistics.append({"frame": frame_index, "n_cells": len(coordinates), "fraction_inside": fraction, "n_outside": len(outside), "expansion": expansion})
    pd.DataFrame(statistics).to_csv(stage_dir / "bound_stats.csv", index=False)

print("Stage-1 traces:", saved)
print("Boundary root:", RUN_ROOT / "bound")


Stage-1 traces: {'T170_to_T171': '<repo>/experiments/Mbrain/artifacts/checkpoints/stage1_res/rollout_stage1_T170_to_T171.npz'}
Boundary root: <repo>/experiments/Mbrain/artifacts/results/bound


In [ ]:
importlib.reload(s2)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

ctx = s2.build_global_ctx(
    adata_path=SCANVI_ADATA,
    lr_pairs_path=lrpr_path,
    ckpt_3dslice=str(STAGE1_CHECKPOINT_ROOT / "T170_to_T171" / "checkpoints" / "best.pt"),        
    device=device,
    layer_col="annotation",
)

stages = []
for sk in fracs:
    src, tgt = sk.split("_to_", 1)
    stages.append(
        s2.StageCfg(
            src=src,
            tgt=tgt,
            out_npz_path=saved[sk],
            bound_dir=f"{resl_path}/bound/{sk}",
            decoder_checkpoint=str(experiment_path(CONFIG["decoder_checkpoint"].format(src=src, tgt=tgt))),
            scanvi_dir=None,
            model_type="scanvi",
            use_lr=True,
            lr_source="decoder",
            use_latent=True,
            latent_key="X_scanVI",
            layer_col="annotation",
        )
    )

outs = s2.run_multi_stages(
    sample_key='sample',
    ctx=ctx,
    stages=stages,
    best_ckpt_dir=str(STAGE2_CHECKPOINT_ROOT),
    train_kwargs=dict(EPOCHS=100, LR=1e-4),
    rl_xy=2.0, rl_z=0.1,
)

for o in outs:
    print(o["best_ckpt_path"])


[Grid] H=208, W=260, H×W=54080
[auto cap] 2 {'src': {'q': 0.9, 'quantile_value': 2.0, 'mean': 1.207677960395813, 'max': 2.0, 'n_cells_used': 28862}, 'tgt': {'q': 0.9, 'quantile_value': 2.0, 'mean': 1.4692920446395874, 'max': 2.0, 'n_cells_used': 31360}, 'cap0': 2, 'capT': 2, 'cap': 2}


Training: 100%|██████████| 100/100 [17:31<00:00, 10.52s/it, rew=-1.0261, best=-1.0209, tgt=0.9574, occ=0.6665]

<repo>/experiments/Mbrain/artifacts/checkpoints/stage2/policy_T170_to_T171.pt


In [ ]:
ckpt_map = {}
for o in outs:
    cfg = o["stage"]
    ckpt_map[f"{cfg.src}_to_{cfg.tgt}"] = o["best_ckpt_path"]

ckpt_map


{'T170_to_T171': '<repo>/experiments/Mbrain/artifacts/checkpoints/stage2/policy_T170_to_T171.pt'}

In [ ]:
importlib.reload(s2)
rollouts = {}
for cfg in stages:
    seg_key = f"{cfg.src}_to_{cfg.tgt}"
    rollouts[seg_key] = s2.rollout_policy_one_stage(
        s2, ctx, cfg, ckpt_map[seg_key], 
        seed=2026, ADVECT_LATENT=True,
        output_dir=Path(resl_path) / 'rollout' / seg_key, output_prefix=seg_key,
    )
    print(seg_key, "Tp1=", len(rollouts[seg_key]["coords"]))


[Grid] H=208, W=260, H×W=54080
[auto cap] 2 {'src': {'q': 0.9, 'quantile_value': 2.0, 'mean': 1.207677960395813, 'max': 2.0, 'n_cells_used': 28862}, 'tgt': {'q': 0.9, 'quantile_value': 2.0, 'mean': 1.4692920446395874, 'max': 2.0, 'n_cells_used': 31360}, 'cap0': 2, 'capT': 2, 'cap': 2}
T170_to_T171 Tp1= 11


## Persist rollout identity metadata in each H5AD frame

This non-lineage model writes only `uid` and `parent_uid` into `adata.obs`. No differentiation fields are created. Existing notebook outputs are preserved.


In [ ]:
ROLLOUT_OBS_FIELDS = ("uid", "parent_uid")

def persist_rollout_obs(rollout_result):
    output_paths = rollout_result.get("output_paths")
    if not isinstance(output_paths, list):
        raise KeyError("rollout_result has no output_paths; pass output_dir to rollout_policy_one_stage")
    if len(output_paths) != len(rollout_result["coords"]):
        raise ValueError("output_paths and rollout frames have different lengths")

    for frame_index, output_path in enumerate(output_paths):
        frame_adata = sc.read_h5ad(output_path)
        for field in ROLLOUT_OBS_FIELDS:
            frames = rollout_result.get(field)
            if not isinstance(frames, list):
                raise KeyError(f"rollout_result does not provide {field!r}")
            values = np.asarray(frames[frame_index])
            if values.ndim != 1 or len(values) != frame_adata.n_obs:
                raise ValueError(
                    f"{field} at frame {frame_index} has shape {values.shape}; "
                    f"expected ({frame_adata.n_obs},)"
                )
            frame_adata.obs[field] = values
        frame_adata.obs_names = frame_adata.obs["uid"].astype(str).to_numpy()
        frame_adata.write_h5ad(output_path, compression="gzip")

    return {"frames_updated": len(output_paths), "obs_fields": list(ROLLOUT_OBS_FIELDS)}

rollout_obs_reports = {
    route: persist_rollout_obs(route_rollout)
    for route, route_rollout in rollouts.items()
}
rollout_obs_reports


In [ ]:
rollouts.keys()


dict_keys(['T170_to_T171'])

In [ ]:

ro = rollouts[seg_key]

for t in range(len(ro["coords"])):
    N  = int(np.asarray(ro["coords"][t]).shape[0])
    nb = int(np.asarray(ro["is_birth"][t]).sum())  
    print(f"{seg_key} | t={t:02d} | N={N} | new_birth={nb}")

npz_path = str(CHECKPOINT_ROOT / 'stage2_res' / 'rollout_stage2_T170_to_T171.npz')
os.makedirs(os.path.dirname(npz_path), exist_ok=True)
np.savez_compressed(npz_path, rollouts=np.array(rollouts, dtype=object))
print("saved:", npz_path)


T170_to_T171 | t=00 | N=34874 | new_birth=0
T170_to_T171 | t=01 | N=35964 | new_birth=5542
T170_to_T171 | t=02 | N=37059 | new_birth=5488
T170_to_T171 | t=03 | N=38161 | new_birth=5719
T170_to_T171 | t=04 | N=39268 | new_birth=5806
T170_to_T171 | t=05 | N=40384 | new_birth=6125
T170_to_T171 | t=06 | N=41508 | new_birth=6311
T170_to_T171 | t=07 | N=42640 | new_birth=6556
T170_to_T171 | t=08 | N=43779 | new_birth=6982
T170_to_T171 | t=09 | N=44928 | new_birth=7589
T170_to_T171 | t=10 | N=46100 | new_birth=8268
saved: <repo>/experiments/Mbrain/artifacts/checkpoints/stage2_res/rollout_stage2_T170_to_T171.npz
